# Binary classification models

## Dependencies

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import random
from collections import defaultdict
import pickle
import torch
import time
import seaborn as sns

In [ ]:
from skimage import io
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import class_weight
from sklearn.metrics import confusion_matrix, f1_score, classification_report

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras import layers, Sequential
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import ConvLSTM2D, BatchNormalization, Conv2D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet import preprocess_input

In [ ]:
random.seed(42)
tf.random.set_seed(42)

## Data

In [ ]:
path_renders = "..\\capturas\\capturas_256x256\\ClassBin"

In [ ]:
database = os.listdir(path_renders)
data_img_labels = database[0]
imgs = database[1:]
print(database[:10])
print(imgs[-1])

In [ ]:
csv_path = os.path.join(path_renders, data_img_labels)

df = pd.read_csv(csv_path, header =0, names=["filename", "correcto"])

# Añadir ruta completa a cada imagen
df["full_path"] = df["filename"].apply(lambda x: os.path.join(path_renders, x))


print(df[["filename", "correcto"]].head())

In [ ]:
for p in imgs[:10]:    
    img_path = os.path.join(path_renders, p)
    print(img_path)

## Auxiliary functions

In [ ]:
def get_paths_and_labels(df, test_size=0.2):
    df_valid = df[df['full_path'].apply(os.path.exists)].copy()
    
    encoder = LabelEncoder()
    df_valid['label_encoded'] = encoder.fit_transform(df_valid['correcto'])
    
    paths = df_valid['full_path'].values
    labels = df_valid['label_encoded'].values
    
    # train test (80/20)
    train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
        paths, labels, test_size=test_size, random_state=42, stratify=labels
    )

    # train val (80/20)
    train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels, test_size=0.2, random_state=42, stratify=train_val_labels
    )

    return (train_paths, train_labels), (val_paths, val_labels), (test_paths, test_labels), encoder


def load_image_tf(path, label, method):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [256, 256])
    
    if method == "resnet":
        img = tf.cast(img, tf.float32) / 255.0
    elif method == "efficientnet":
        img = tf.cast(img, tf.float32) 
    elif method == "mobilenet":
        img = (tf.cast(img, tf.float32) / 127.5) - 1.0
    return img, label

In [ ]:
def plot_from_history(history, epochs):
    hist = history.history if hasattr(history, 'history') else history

    # Determinar cuántas métricas tenemos para ajustar el tamaño de la figura
    has_precision_recall = 'precision' in hist and 'recall' in hist
    rows = 2 if has_precision_recall else 1
    
    plt.figure(figsize=(12, 4 * rows))
    epochs_range = range(len(hist['loss'])) # Usar el largo real de los datos

    # --- 1. Accuracy ---
    plt.subplot(rows, 2, 1)
    plt.plot(epochs_range, hist['accuracy'], label='Train Accuracy')
    plt.plot(epochs_range, hist['val_accuracy'], label='Val Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.legend(loc='lower right')

    # --- 2. Loss ---
    plt.subplot(rows, 2, 2)
    plt.plot(epochs_range, hist['loss'], label='Train Loss')
    plt.plot(epochs_range, hist['val_loss'], label='Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.legend(loc='upper right')

    # --- 3. Precision y Recall ---
    if has_precision_recall:
        # Precision
        plt.subplot(rows, 2, 3)
        # Nota: Keras a veces nombra la métrica 'precision_1' etc., buscamos la clave
        p_key = [k for k in hist.keys() if 'precision' in k and 'val' not in k][0]
        plt.plot(epochs_range, hist[p_key], label='Train Precision', color='green')
        plt.plot(epochs_range, hist[f'val_{p_key}'], label='Val Precision', color='lightgreen')
        plt.title('Training and Validation Precision')
        plt.xlabel('Epochs')
        plt.legend(loc='lower right')

        # Recall
        plt.subplot(rows, 2, 4)
        r_key = [k for k in hist.keys() if 'recall' in k and 'val' not in k][0]
        plt.plot(epochs_range, hist[r_key], label='Train Recall', color='purple')
        plt.plot(epochs_range, hist[f'val_{r_key}'], label='Val Recall', color='violet')
        plt.title('Training and Validation Recall')
        plt.xlabel('Epochs')
        plt.legend(loc='lower right')

    plt.tight_layout()
    plt.show()

In [ ]:
def show_confusion_matrix(y_true, y_pred, class_names):

    cm = confusion_matrix(y_true, y_pred)

    tn, fp, fn, tp = cm.ravel()

    print("=== MATRIZ DE CONFUSIÓN BINARIA ===")
    print(f"{'Metrica':<20} | {'Valor':<10} | {'Interpretación'}")
    print("-" * 60)

    # Función auxiliar para imprimir filas
    def print_row(label, val, desc):
        print(f"{label:<20} | {val:<10.0f} | {desc}")

    print_row("Verdaderos Neg. (TN)", tn, f"Correctos: {class_names[0]}")
    print_row("Falsos Pos. (FP)", fp, f"Error: Era {class_names[0]} pero predijo {class_names[1]}")
    print_row("Falsos Neg. (FN)", fn, f"Error: Era {class_names[1]} pero predijo {class_names[0]}")
    print_row("Verdaderos Pos. (TP)", tp, f"Correctos: {class_names[1]}")

    print("-" * 60)

    # Resumen de estado
    total_clase_positiva = tp + fn
    umbral = total_clase_positiva * 0.25

    print(f"Estado de la clase {class_names[1]}: ", end="")
    if fn == 0 and fp == 0:
        print("Perfecto")
    elif fn > umbral:
        print(f"Muchos Falsos Negativos: {fn}")
    elif fp > umbral:
        print(f"Muchos Falsos Positivos: {fp}")
    else:
        print("OK")


In [ ]:
def save_history(history, nombre_archivo):
    with open(nombre_archivo, "wb") as f:
        pickle.dump(history.history, f)

def load_history(nombre_archivo):
    if os.path.exists(nombre_archivo):
        with open(nombre_archivo, "rb") as f:
            loaded_history = pickle.load(f)
        print(f"Historial cargado desde {nombre_archivo}.")
        return loaded_history
    else:
        print(f"No se encontró el archivo {nombre_archivo}.")
        return None

In [ ]:
def training_pipeline_db(model, train_ds, val_ds, test_ds, patience=3, epochs=10, lr=0.005):
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=patience, verbose=1, min_lr=4e-10
    )

    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]    )

    history = model.fit(
        train_ds, 
        epochs=epochs, 
        validation_data=val_ds, 
        callbacks=[lr_callback]
    )

    print("\nEvaluando en Entrenamiento...")
    results_train = model.evaluate(train_ds, verbose=0)
    train_loss = results_train[0]
    train_acc = results_train[1]

    print("Evaluando en Test...")
    results_test = model.evaluate(test_ds, verbose=0)
    test_loss = results_test[0]
    test_acc = results_test[1]
    
    return test_acc, train_acc, train_loss, test_loss, history

## Get Dataset

In [ ]:
(train_paths, train_labels), (val_paths, val_labels), (test_paths, test_labels), encoder = get_paths_and_labels(df, test_size=0.2)

print(encoder.classes_)
for i, nombre in enumerate(encoder.classes_):
    print(f"ID {i} -> Clase: {nombre}")

print(f"Total: {len(train_paths) + len(val_paths) + len(test_paths)}")
print(f"Entrenamiento: {len(train_paths)}")
print(f"Validación:    {len(val_paths)}")
print(f"Test:          {len(test_paths)}")

print("\nDistribución de clases en el dataset completo:")
print(df['correcto'].value_counts())
print(df['correcto'].value_counts(normalize=True) * 100)

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='correcto', data=df, palette='viridis')
plt.title('Distribución de Imágenes por Clase')
plt.xticks(rotation=45)
plt.show()

## Models

In [ ]:
resultados_tiempo = {}

### MobileNet

In [ ]:
from tensorflow.keras.applications import MobileNet

In [ ]:
# Crear dataset de entrenamiento
train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = (train_ds
            .shuffle(len(train_paths)) # Shuffle de rutas
            .map(lambda x, y: load_image_tf(x, y, method="mobilenet"), num_parallel_calls=tf.data.AUTOTUNE) # Carga paralela
            .batch(32)
            .prefetch(tf.data.AUTOTUNE))

# Crear dataset de validación
val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds = (val_ds
          .map(lambda x, y: load_image_tf(x, y, method="mobilenet"), num_parallel_calls=tf.data.AUTOTUNE)
          .batch(32)
          .prefetch(tf.data.AUTOTUNE))

# Crear dataset de test
test_ds = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
test_ds = (test_ds
           .map(lambda x, y: load_image_tf(x, y, method="mobilenet"), num_parallel_calls=tf.data.AUTOTUNE)
           .batch(32)
           .prefetch(tf.data.AUTOTUNE))

In [ ]:
y_true_list = []
for images, labels in test_ds:
    y_true_list.append(labels.numpy())

y_true = np.concatenate(y_true_list, axis=0)

In [ ]:
start_time = time.time()

base_mobilenet = MobileNet(
    include_top = False,
    weights = None,
    input_shape = (256, 256, 3),
    pooling = "avg"
)

model_bin_mobilenet = Sequential([
    base_mobilenet,
    layers.Dense(1, activation='sigmoid')   # 2 clases
])

epochs = 10

test_acc_mobilenet, train_acc_mobilenet, train_loss_mobilenet, test_loss_mobilenet, history_bin_mobilenet = training_pipeline_db(model_bin_mobilenet, train_ds, val_ds, test_ds, epochs=epochs, lr = 0.0001)
print(f"Test accuracy: {test_acc_mobilenet}\nTrain accuracy: {train_acc_mobilenet}")
print(f"Test loss: {test_loss_mobilenet}\nTrain loss: {train_loss_mobilenet}")

end_time = time.time()
training_time = end_time - start_time
resultados_tiempo['MobileNet'] = training_time
print(f"Tiempo total de entrenamiento: {training_time:.2f} segundos")
print(f"Tiempo total de entrenamiento: {training_time / 60:.2f} minutos")

In [ ]:
plot_from_history(history_bin_mobilenet, epochs = epochs)

In [ ]:
model_bin_mobilenet.save("model_bin_mobilenet.keras")
save_history(history_bin_mobilenet, "history_bin_mobilenet.pkl")

In [ ]:
model_bin_mobilenet = load_model("model_bin_mobilenet.keras")
history_bin_mobilenet = load_history("history_bin_mobilenet.pkl")

In [ ]:
y_pred_mobilenet = model_bin_mobilenet.predict(test_ds)
y_pred_classes_mobilenet = (y_pred_mobilenet > 0.5).astype(int).flatten()
class_names = encoder.classes_ 
class_names = [str(name) for name in class_names]

acc_mobilenet = accuracy_score(y_true, y_pred_classes_mobilenet)
print(f"Accuracy MobileNet: {acc_mobilenet}")
print("-------------------------------------")

f1_mobilenet = f1_score(y_true, y_pred_classes_mobilenet, average="weighted")
print(f"F1 Score MobileNet: {f1_mobilenet}")
print("-------------------------------------")

print("Classification Report MobileNet:")
print(classification_report(y_true, y_pred_classes_mobilenet, target_names=class_names))

In [ ]:
show_confusion_matrix(y_true, y_pred_classes_mobilenet, class_names)

Train, test y val con datos capturados desde Blender

| Métrica                | Valor | Interpretación                              |
|------------------------|------:|---------------------------------------------|
| Verdaderos Neg. (TN)   |    | Correctos: False                           |
| Falsos Pos. (FP)       |      | Error: Era False pero predijo True         |
| Falsos Neg. (FN)       |      | Error: Era True pero predijo False         |
| Verdaderos Pos. (TP)   |    | Correctos: True                            |                                         

In [ ]:
def get_real_test_dataset():
    path =  "../capturas/capturas_256x256/capturasVR"
    database = os.listdir(path)
    data_img_labels = database[0]

    csv_path = os.path.join(path, data_img_labels)

    df = pd.read_csv(csv_path, header =0, names=["filename", "label"])

    df["full_path"] = df["filename"].apply(lambda x: os.path.join(path, x))
    df_valid = df[df['full_path'].apply(os.path.exists)].copy()
    
    encoder = LabelEncoder()
    df_valid['label_encoded'] = encoder.fit_transform(df_valid['label'])
    
    df_class_2 = df_valid[df_valid['label_encoded'] == 2].copy()
    df_class_2['label_binary'] = True   
    
    paths = df_class_2['full_path'].values
    labels = df_class_2['label_binary'].values

    return tf.data.Dataset.from_tensor_slices((paths, labels))

In [ ]:
real_test_ds = get_real_test_dataset()
real_test_ds = (real_test_ds
           .map(lambda x, y: load_image_tf(x, y, method="mobilenet"), num_parallel_calls=tf.data.AUTOTUNE)
           .batch(32)
           .prefetch(tf.data.AUTOTUNE))

y_true_list = []
for images, labels in real_test_ds:
    y_true_list.append(labels.numpy())

y_true_real = np.concatenate(y_true_list, axis=0)

y_pred_real = model_bin_mobilenet.predict(real_test_ds)
y_pred_classes_real= (y_pred_real > 0.5).astype(int).flatten()

class_names_real = encoder.classes_ 
class_names_real = [str(name) for name in class_names_real]

acc_mobilenet = accuracy_score(y_true_real, y_pred_classes_real)
print(f"Accuracy MobileNet: {acc_mobilenet}")
print("-------------------------------------")

f1_mobilenet = f1_score(y_true_real, y_pred_classes_real, average="weighted")
print(f"F1 Score MobileNet: {f1_mobilenet}")
print("-------------------------------------")

print("Classification Report MobileNet:")
print(classification_report(y_true_real, y_pred_classes_real, target_names=class_names_real))


In [ ]:
show_confusion_matrix(y_true_real, y_pred_classes_real, class_names_real)